# 10 · Three vendors, three message formats
Level L2 · Part 2 · professional · ~35 min · builds on 08

## Goal
Write the same two-turn Scout conversation as an Anthropic Messages, an OpenAI Responses and a Gemini
Interactions request, prove each vendor's own SDK accepts it, and name the four things that differ.

## The idea
Every chat API takes the same three things: the developer's rules, the turns so far, and a cap on the
reply. Lesson 08 sent them to Gemini's `generate_content`. The three current flagship APIs each package
them differently: where the rules go, what the roles are called, how a piece of text is wrapped, and
what the length cap is named. None of it is hard, but a wrong guess is a 400 error or, worse, a silently
ignored instruction. So Scout keeps one vendor-neutral history and converts it at the edge, one small
function per vendor.

Analogy: the same letter posted in three countries. The words do not change; the envelope layout does.

## Picture
```mermaid
flowchart LR
    N["Scout history<br/>{role, text} list"] --> A["to_anthropic<br/>Messages"]
    N --> O["to_responses<br/>OpenAI Responses"]
    N --> G["to_interactions<br/>Gemini Interactions"]
    A --> W["each vendor SDK<br/>builds the HTTP body"]
    O --> W
    G --> W
```

## Step by step
### 1. Start the recorder
Every HTTP request in this notebook is recorded once to `../data/llm_cassettes/10_three-message-formats.yaml`
and replayed after that, with no key.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("10_three-message-formats")

LLM mode: replay (10_three-message-formats.yaml)


### 2. Setup from lesson 08
Scout's notes and system text, the Gemini client on Vertex, and lesson 08's neutral-format helpers:
`check_messages` (the validity rules) and `to_gemini` (the `generate_content` converter). Nothing new.

In [2]:
import os, json
from dotenv import load_dotenv
from google import genai
from google.genai import types

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
NOTES_TEXT = "\n".join(f"Note {n['id']}: {n['text']}" for n in NOTES)
SYSTEM_TEXT = ("You are Scout, a research assistant. Answer only from the user's notes below and this chat. "
               "If the notes do not cover it, say so in one sentence. Keep answers to one sentence.\n\n" + NOTES_TEXT)
load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)
print(len(NOTES), "notes; Gemini model:", MODEL)

6 notes; Gemini model: gemini-3.5-flash-lite


The two helpers, copied from lesson 08.

In [3]:
def turn(role: str, text: str) -> types.Content:
    return types.Content(role=role, parts=[types.Part(text=text)])

def to_gemini(messages: list[dict]) -> tuple[str | None, list[types.Content]]:
    system = messages[0]["text"] if messages and messages[0]["role"] == "system" else None
    rest = messages[1:] if system is not None else messages
    return system, [turn({"user": "user", "assistant": "model"}[m["role"]], m["text"]) for m in rest]

def check_messages(messages: list[dict]) -> None:
    if not messages:
        raise ValueError("message list is empty")
    for i, m in enumerate(messages):
        if m["role"] not in ("system", "user", "assistant"):
            raise ValueError(f"message {i}: unknown role {m['role']!r}")
        if m["role"] == "system" and i != 0:
            raise ValueError(f"message {i}: system message must come first")
        if not m["text"].strip():
            raise ValueError(f"message {i}: empty text")
    if messages[-1]["role"] != "user":
        raise ValueError(f"last message must be from the user, not {messages[-1]['role']!r}")
print("helpers ready")

helpers ready


### 3. Turn 1, on the real model
A two-turn conversation needs a real first reply to put in the history. We get it from Gemini (the
one vendor with a key on this machine), exactly as lesson 08 did.

In [4]:
q1 = "Hi, I'm Asha. I'm researching Mars rovers."
system, contents = to_gemini([{"role": "system", "text": SYSTEM_TEXT}, {"role": "user", "text": q1}])
config = types.GenerateContentConfig(system_instruction=system, max_output_tokens=60,
                                     automatic_function_calling=NO_AFC)
first_reply = client.models.generate_content(model=MODEL, contents=contents, config=config).text.strip()
print("Scout:", first_reply)

Scout: Hello Asha, my notes show that the rover Perseverance landed on Mars in February 2021.


Now the neutral history: system rules, turn 1, Scout's real reply, and the turn-2 question. (If the
model cell above was skipped, a labelled placeholder stands in for the reply so the rest still runs.)

In [5]:
if "first_reply" not in globals():
    first_reply = "(placeholder: Scout's turn-1 reply, model cell skipped)"
chat = [
    {"role": "system", "text": SYSTEM_TEXT},
    {"role": "user", "text": q1 if "q1" in globals() else "Hi, I'm Asha. I'm researching Mars rovers."},
    {"role": "assistant", "text": first_reply},
    {"role": "user", "text": "Which of my notes is about a rover?"},
]
check_messages(chat)
for m in chat:
    print(f"{m['role']:>9}: {m['text'][:60]!r}")

   system: 'You are Scout, a research assistant. Answer only from the us'
     user: "Hi, I'm Asha. I'm researching Mars rovers."
assistant: 'Hello Asha, my notes show that the rover Perseverance landed'
     user: 'Which of my notes is about a rover?'


### 4. Anthropic Messages
Anthropic's API is `client.messages.create`. Three things to see:
- **system** is a top-level parameter, not a message. `messages` holds only `user` and `assistant`.
- each message's `content` is a list of **content blocks**, here `{"type": "text", "text": ...}`.
  (A plain string is accepted as a shortcut; blocks are what you need once images or tool results arrive.)
- **`max_tokens` is required.**

In [6]:
VENDOR_MODELS = {"anthropic": "claude-sonnet-5-5", "openai": "gpt-6-luna", "gemini": MODEL}

def to_anthropic(messages: list[dict], max_tokens: int = 60) -> dict:
    check_messages(messages)
    system = messages[0]["text"] if messages[0]["role"] == "system" else None
    rest = messages[1:] if system is not None else messages
    payload = {"model": VENDOR_MODELS["anthropic"], "max_tokens": max_tokens,
               "messages": [{"role": m["role"], "content": [{"type": "text", "text": m["text"]}]}
                            for m in rest]}
    if system is not None:
        payload["system"] = system
    return payload

anthropic_req = to_anthropic(chat)
print(json.dumps(anthropic_req["messages"][1], indent=1)[:200])

{
 "role": "assistant",
 "content": [
  {
   "type": "text",
   "text": "Hello Asha, my notes show that the rover Perseverance landed on Mars in February 2021."
  }
 ]
}


### 5. OpenAI Responses
OpenAI's current API is `client.responses.create` (Chat Completions is "not recommended for new
projects"). The history goes in `input`, a list of **items**:
- the rules go in a message with the **developer role** (or, equivalently, the `instructions` parameter);
- user text is a content part of type `input_text`; an earlier assistant reply can be a plain string;
- the cap is called `max_output_tokens` and is optional.

In [7]:
def to_responses(messages: list[dict], max_output_tokens: int = 60) -> dict:
    check_messages(messages)
    items = []
    for m in messages:
        if m["role"] == "system":
            items.append({"role": "developer", "content": m["text"]})
        elif m["role"] == "user":
            items.append({"role": "user", "content": [{"type": "input_text", "text": m["text"]}]})
        else:
            items.append({"role": "assistant", "content": m["text"]})
    return {"model": VENDOR_MODELS["openai"], "input": items, "max_output_tokens": max_output_tokens}

responses_req = to_responses(chat)
print([item["role"] for item in responses_req["input"]])

['developer', 'user', 'assistant', 'user']


### 6. Gemini Interactions
Gemini's recommended API is now `client.interactions.create` (`generate_content`, which we used so far,
is "fully supported but legacy"). Here the history is a list of **steps**, not messages:
- a user turn is a `user_input` step, the model's earlier reply a `model_output` step (no `role` field);
- content items are `{"type": "text", "text": ...}`;
- the rules are `system_instruction` and the cap sits inside `generation_config`.

In [8]:
def to_interactions(messages: list[dict], max_output_tokens: int = 60) -> dict:
    check_messages(messages)
    system = messages[0]["text"] if messages[0]["role"] == "system" else None
    rest = messages[1:] if system is not None else messages
    steps = [{"type": "user_input" if m["role"] == "user" else "model_output",
              "content": [{"type": "text", "text": m["text"]}]} for m in rest]
    payload = {"model": VENDOR_MODELS["gemini"], "input": steps,
               "generation_config": {"max_output_tokens": max_output_tokens}}
    if system is not None:
        payload["system_instruction"] = system
    return payload

interactions_req = to_interactions(chat)
print([step["type"] for step in interactions_req["input"]])

['user_input', 'model_output', 'user_input']


### 7. Side by side
The same four facts, read out of each payload. `shape` prints where the rules went, the label of each
turn, the first turn's text wrapper and the reply cap.

In [9]:
def shape(name: str, req: dict, turns_key: str, label_key: str, rules_key: str) -> None:
    turns = [t for t in req[turns_key] if t.get(label_key) != "developer"]
    cap = {k: v for k, v in req.items() if "max" in k} or req.get("generation_config")
    first = turns[0]["content"]
    print(f"{name:13}| rules in: {rules_key:18}| turns: {[t[label_key] for t in turns]}")
    print(f"{'':13}| text wrapper: {json.dumps(first[0] if isinstance(first, list) else first)[:40]:42}| cap: {cap}")

shape("Anthropic", anthropic_req, "messages", "role", "system")
shape("OpenAI", responses_req, "input", "role", "developer message")
shape("Gemini Inter.", interactions_req, "input", "type", "system_instruction")

Anthropic    | rules in: system            | turns: ['user', 'assistant', 'user']


             | text wrapper: {"type": "text", "text": "Hi, I'm Asha.   | cap: {'max_tokens': 60}
OpenAI       | rules in: developer message | turns: ['user', 'assistant', 'user']
             | text wrapper: {"type": "input_text", "text": "Hi, I'm   | cap: {'max_output_tokens': 60}
Gemini Inter.| rules in: system_instruction| turns: ['user_input', 'model_output', 'user_input']
             | text wrapper: {"type": "text", "text": "Hi, I'm Asha.   | cap: {'max_output_tokens': 60}


The words are identical in all three; only the envelope changed. The table below sums it up, with
lesson 08's `generate_content` for comparison.

| | Anthropic Messages | OpenAI Responses | Gemini Interactions | Gemini `generate_content` |
|---|---|---|---|---|
| rules | top-level `system` | `developer` item or `instructions` | `system_instruction` | config `system_instruction` |
| turn labels | role `user` / `assistant` | role `user` / `assistant` | step `user_input` / `model_output` | role `user` / `model` |
| text wrapper | block `{"type": "text"}` | part `{"type": "input_text"}` | content `{"type": "text"}` | `Part(text=...)` |
| reply cap | `max_tokens` (required) | `max_output_tokens` | `generation_config.max_output_tokens` | config `max_output_tokens` |

### 8. Do the SDKs accept these payloads?
We have no Anthropic or OpenAI key, and the Interactions API is not on our Vertex key (lesson 09). We can
still check the payloads against the real SDKs: give each SDK an HTTP client whose transport answers
every request with a local 400 "not sent". The SDK builds the real request; we read its URL and JSON body.
Nothing leaves the machine.

The three SDKs now use `httpx2` (a fork of `httpx`) as their HTTP layer, so a custom client must come
from `httpx2`; a plain `httpx` client would not be accepted. Mocking here is only a checking tool;
testing agents with mocks is taught properly in lesson 46.

In [10]:
import httpx2, anthropic, openai
NO_KEY = "unused"                                  # nothing is sent, so no real key is needed

def refuse(request: httpx2.Request) -> httpx2.Response:
    return httpx2.Response(400, json={"error": {"type": "invalid_request_error", "message": "not sent"}})

def local_http() -> httpx2.Client:
    return httpx2.Client(transport=httpx2.MockTransport(refuse))

def wire(send) -> dict:
    try:
        send()
    except Exception as err:                       # each SDK raises its own 400 error class
        req = err.response.request
        print(req.method, req.url)
        return json.loads(req.content)
    raise RuntimeError("expected the local 400")

Anthropic first. The SDK takes our payload as keyword arguments.

In [11]:
claude = anthropic.Anthropic(api_key=NO_KEY, max_retries=0, http_client=local_http())
sent = wire(lambda: claude.messages.create(**anthropic_req))
print(sorted(sent), "| same as ours:", sent == anthropic_req)

POST https://api.anthropic.com/v1/messages
['max_tokens', 'messages', 'model', 'system'] | same as ours: True


Then OpenAI Responses and Gemini Interactions (a Gemini Developer API client, `vertexai` off).

In [12]:
gpt = openai.OpenAI(api_key=NO_KEY, max_retries=0, http_client=local_http())
sent = wire(lambda: gpt.responses.create(**responses_req))
print(sorted(sent), "| same as ours:", sent == responses_req)

gem = genai.Client(api_key=NO_KEY, http_options={"httpx_client": local_http()})
sent = wire(lambda: gem.interactions.create(**interactions_req))
print(sorted(sent), "| same as ours:", sent == interactions_req)

POST https://api.openai.com/v1/responses
['input', 'max_output_tokens', 'model'] | same as ours: True


POST https://generativelanguage.googleapis.com/v1beta/interactions
['generation_config', 'input', 'model', 'system_instruction'] | same as ours: True


Three different endpoints, and each body is exactly our dict: the SDKs are thin wrappers over these JSON
shapes. This proves the payloads are well formed for the SDK, not that the server will like every value
(for example a retired model id). Only a real call shows that.

### 9. Turn 2, for the vendor we have a key for
On this machine that is Gemini on Vertex, through `generate_content`. Same neutral history, fourth converter.

In [13]:
system, contents = to_gemini(chat)
config = types.GenerateContentConfig(system_instruction=system, max_output_tokens=60,
                                     automatic_function_calling=NO_AFC)
resp = client.models.generate_content(model=MODEL, contents=contents, config=config)
print("Scout:", resp.text.strip())
print("input tokens:", resp.usage_metadata.prompt_token_count, "| finish:", resp.candidates[0].finish_reason)

Scout: Note 5 is about a rover.
input tokens: 186 | finish: FinishReason.STOP


With your own keys, the other three calls are one line each, and the reply text sits in a different
place each time (read-along, not run here: no keys):
```python
msg = anthropic.Anthropic().messages.create(**anthropic_req)
msg.content[0].text, msg.stop_reason, msg.usage.input_tokens      # a list of content blocks again
r = openai.OpenAI().responses.create(**responses_req)
r.output_text, r.usage.input_tokens                               # output_text joins the output items
i = genai.Client(api_key=GEMINI_API_KEY).interactions.create(**interactions_req)
i.output_text, i.usage.total_input_tokens                         # the reply is a model_output step
```

## At work
- **Teams put one layer of request adapters between the app and the vendors,** like our four converters, so a model
  swap is a config change. Gateways and libraries (LiteLLM and others) sell exactly this. → lesson 37
- **The formats drift.** OpenAI moved from Chat Completions to Responses and shut the Assistants API on
  2026-08-26; Gemini made Interactions the default in June 2026. Code pinned to old shapes keeps working
  until a sunset date, then breaks. → lesson 17
- **Tool calls differ even more than text** (Anthropic `tool_use` blocks, OpenAI `function_call` items,
  Gemini `function_call` steps), and that is where most request-adapter bugs live. → lesson 23
- **Request mocks like our `refuse` transport are how teams test vendor code in CI** without keys or cost. → lesson 46

## What just happened
- One neutral `{"role", "text"}` history (with a real Gemini reply in it) became three vendor payloads.
- Anthropic Messages: top-level `system`, `user`/`assistant` messages of content blocks, required `max_tokens`.
- OpenAI Responses: `input` items, the rules as a `developer` message, `input_text` parts, optional `max_output_tokens`.
- Gemini Interactions: `user_input` / `model_output` steps, `system_instruction`, cap inside `generation_config`.
- Each vendor's SDK, given a local transport, produced exactly our dict as the request body.
- Turn 2 ran for real on Gemini (`generate_content`), the one vendor with a key here.

## Common mistakes
**1. Leaving out `max_tokens` on Anthropic.** OpenAI and Gemini make the cap optional; Anthropic's SDK
refuses before sending anything.

In [14]:
no_cap = {k: v for k, v in anthropic_req.items() if k != "max_tokens"}
claude.messages.create(**no_cap)

TypeError: Missing required arguments; Expected either ('max_tokens', 'messages' and 'model') or ('max_tokens', 'messages', 'model' and 'stream') arguments to be given

Fix: always set the cap; `to_anthropic` does it for you.

**2. Using a Chat Completions habit on Responses.** Old tutorials pass `messages=`. The Responses method
has no such parameter.

In [15]:
gpt.responses.create(model=VENDOR_MODELS["openai"], messages=[{"role": "user", "content": "Hi"}])

TypeError: Responses.create() got an unexpected keyword argument 'messages'

Fix: the turns go in `input` (and the rules in a `developer` item or `instructions`), as `to_responses` builds them.

## Try it
Write `from_anthropic(payload)` that turns an Anthropic payload back into Scout's neutral list (system
first if present, then one `{"role", "text"}` per message, joining the text blocks). Check that
`from_anthropic(to_anthropic(chat)) == chat`. Then write `from_interactions` the same way and check its
round trip too.

In [16]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [17]:
llm_replay.stop()

## Key terms
- **Messages API** — Anthropic's chat API (`client.messages.create`): top-level `system`, `user`/`assistant` messages made of content blocks, required `max_tokens`.
- **Responses API** — OpenAI's current API (`client.responses.create`): a list of `input` items, rules as a `developer` message or `instructions`, `max_output_tokens`.
- **Interactions API** — Gemini's recommended API (`client.interactions.create`, Gemini Developer API): history as `user_input`/`model_output` steps, `system_instruction`, `generation_config`.
- developer role: see lesson 08.
- **content block** — one typed piece of a message, such as `{"type": "text", "text": ...}`; images, tool calls and tool results are other block types.
- **request adapter** — a small function that converts an app's own message format into one vendor's request format.